In [ ]:
!pip install faiss-cpu

In [ ]:
import os
import ast
import glob
import warnings
from typing import List, Tuple, Optional, Union

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
from transformers import CLIPProcessor, CLIPModel
import faiss
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("⚠️  No GPU. Using CPU.")

print(f"🖥️  Device: {DEVICE}")
print("✅ Section 1 COMPLETE")

In [ ]:
CANDIDATES = [
    "/kaggle/input/deepfashion2-original-with-dataframes/DeepFashion2",
    "/kaggle/input/datasets/thusharanair/deepfashion2-original-with-dataframes/DeepFashion2",
]

BASE_PATH = None
for c in CANDIDATES:
    if os.path.isdir(c):
        BASE_PATH = c
        break

if BASE_PATH is None:
    for item in os.listdir("/kaggle/input/"):
        test = os.path.join("/kaggle/input/", item, "DeepFashion2")
        if os.path.isdir(test):
            BASE_PATH = test
            break

if BASE_PATH is None:
    raise FileNotFoundError("DeepFashion2 not found. Add the dataset first.")

IMAGE_PATH     = os.path.join(BASE_PATH, "deepfashion2_original_images", "train", "image")
DATAFRAME_PATH = os.path.join(BASE_PATH, "img_info_dataframes")
TRAIN_DF_PATH  = os.path.join(DATAFRAME_PATH, "train.csv")

print("=" * 55)
for name, path in [("BASE_PATH", BASE_PATH), ("IMAGE_PATH", IMAGE_PATH),
                    ("DATAFRAME_PATH", DATAFRAME_PATH), ("TRAIN_DF_PATH", TRAIN_DF_PATH)]:
    icon = "✅" if os.path.exists(path) else "❌"
    print(f"  {icon} {name}: {path}")

test_image = os.path.join(IMAGE_PATH, "000001.jpg")
if os.path.exists(test_image):
    print(f"\n  🖼️  Sample image verified: {test_image}")
else:
    test_image2 = os.path.join(IMAGE_PATH, "000002.jpg")
    test_image3 = os.path.join(IMAGE_PATH, "000010.jpg")
    if os.path.exists(test_image2):
        print(f"\n  🖼️  Sample image verified: {test_image2}")
    elif os.path.exists(test_image3):
        print(f"\n  🖼️  Sample image verified: {test_image3}")
    else:
        print(f"\n  ⚠️  Could not verify sample image (path may need adjustment in Section 3)")

print("=" * 55)
print("✅ Section 2 COMPLETE")

In [ ]:
print("📂 Loading train.csv...")
train_df = pd.read_csv(TRAIN_DF_PATH)

print(f"✅ Loaded {len(train_df):,} rows")
print(f"📋 Columns: {list(train_df.columns)}")
print(f"\n📊 First 3 rows:")
print(train_df[['path', 'b_box', 'category_id', 'category_name']].head(3))

PATH_COL = 'path'
BBOX_COL = 'b_box' 

print(f"\n✅ Path column: '{PATH_COL}'")
print(f"✅ BBox column: '{BBOX_COL}'")
print(f"   Sample bbox: {train_df[BBOX_COL].iloc[0]}")

raw_path = str(train_df[PATH_COL].iloc[0])
print(f"\n🔍 Raw path from CSV: {raw_path}")
print(f"   Exists? {os.path.exists(raw_path)}")

if os.path.exists(raw_path):
    train_df["full_path"] = train_df[PATH_COL].apply(str)
    print("✅ Paths in CSV are already valid — using as-is")
else:

    OLD_PREFIX = "/kaggle/input/deepfashion2-original-with-dataframes/DeepFashion2"
    NEW_PREFIX = BASE_PATH 

    print(f"\n🔧 Fixing path prefix:")
    print(f"   OLD: {OLD_PREFIX}")
    print(f"   NEW: {NEW_PREFIX}")

    train_df["full_path"] = train_df[PATH_COL].apply(
        lambda x: str(x).replace(OLD_PREFIX, NEW_PREFIX)
    )

    fixed_path = train_df["full_path"].iloc[0]
    print(f"\n   Fixed path: {fixed_path}")
    print(f"   Exists? {os.path.exists(fixed_path)}")

    if not os.path.exists(fixed_path):
        print("\n🔧 Trying filename-only approach...")
        train_df["full_path"] = train_df[PATH_COL].apply(
            lambda x: os.path.join(IMAGE_PATH, os.path.basename(str(x)))
        )
        fixed_path2 = train_df["full_path"].iloc[0]
        print(f"   Fixed path: {fixed_path2}")
        print(f"   Exists? {os.path.exists(fixed_path2)}")

print(f"\n🔍 Verifying 5 random files...")
check_indices = [0, 100, 1000, 5000, 10000]
found = 0
for i in check_indices:
    if i < len(train_df):
        exists = os.path.exists(train_df["full_path"].iloc[i])
        if exists:
            found += 1

print(f"   Result: {found}/5 files found")

print(f"\n📊 Category distribution:")
print(train_df['category_name'].value_counts().to_string())

print(f"\n{'='*55}")
print(f"✅ Section 3 COMPLETE — {len(train_df):,} records loaded")
print(f"   PATH_COL = '{PATH_COL}'")
print(f"   BBOX_COL = '{BBOX_COL}' (bounding box: [x1, y1, x2, y2])")
print(f"   Files verified: {found}/5")
print(f"{'='*55}")

In [ ]:
SUBSET_SIZE = 10000

subset_df = train_df.sample(n=SUBSET_SIZE, random_state=42).reset_index(drop=True)

print(f"📊 Subset created:")
print(f"   Original dataset:  {len(train_df):,} rows")
print(f"   Selected subset:   {len(subset_df):,} rows")
print(f"   Random state:      42 (reproducible)")

verified = sum(1 for i in [0, 100, 5000] if os.path.exists(subset_df["full_path"].iloc[i]))
print(f"   Files verified:    {verified}/3")

print(f"\n📊 Subset category distribution:")
print(subset_df['category_name'].value_counts().to_string())

print(f"\n{'='*55}")
print(f"✅ Section 4 COMPLETE — {len(subset_df):,} images ready")
print(f"{'='*55}")

In [ ]:
def parse_bbox(bbox_value):
    """Convert bbox from string or list to [x1, y1, x2, y2] integers."""
    if isinstance(bbox_value, (list, tuple)):
        return [int(float(v)) for v in bbox_value]
    if isinstance(bbox_value, str):
        try:
            parsed = ast.literal_eval(bbox_value)
            return [int(float(v)) for v in parsed]
        except:
            cleaned = bbox_value.strip('[]() ').replace(' ', '')
            return [int(float(v)) for v in cleaned.split(',')]
    return None

def crop_clothing(image_path, bbox_value, padding=10):
    """
    Crop clothing region from image using bounding box.
    Returns cropped PIL Image, or full image if bbox fails.
    """
    try:
        img = Image.open(image_path).convert("RGB")
        w, h = img.size

        if bbox_value is None:
            return img
        if isinstance(bbox_value, float) and np.isnan(bbox_value):
            return img

        bbox = parse_bbox(bbox_value)
        if bbox is None or len(bbox) < 4:
            return img

        x1, y1, x2, y2 = bbox
        x1 = max(0, x1 - padding)
        y1 = max(0, y1 - padding)
        x2 = min(w, x2 + padding)
        y2 = min(h, y2 + padding)

        if x2 <= x1 or y2 <= y1:
            return img

        return img.crop((x1, y1, x2, y2))

    except Exception as e:
        return None

print("🔍 Testing cropping on 10 images...")
success = 0
for i in range(10):
    row = subset_df.iloc[i]
    result = crop_clothing(row['full_path'], row[BBOX_COL])
    if result is not None:
        success += 1
print(f"   Result: {success}/10 successful crops")

print("\n🖼️  Showing Original vs Cropped:")

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("Top Row: Original  |  Bottom Row: Cropped Clothing Region",
             fontsize=14, fontweight='bold')

for i in range(5):
    row = subset_df.iloc[i]
    try:
        original = Image.open(row['full_path']).convert("RGB")
        cropped = crop_clothing(row['full_path'], row[BBOX_COL])

        axes[0, i].imshow(original)
        axes[0, i].set_title(f"{row['category_name']}", fontsize=9)
        axes[0, i].axis('off')

        if cropped:
            axes[1, i].imshow(cropped)
            bbox = parse_bbox(row[BBOX_COL])
            axes[1, i].set_title(f"bbox: {bbox}", fontsize=8)
        axes[1, i].axis('off')

    except:
        axes[0, i].axis('off')
        axes[1, i].axis('off')

plt.tight_layout()
plt.show()

print(f"\n{'='*55}")
print(f"✅ Section 5 COMPLETE — Cropping pipeline ready")
print(f"{'='*55}")

In [ ]:
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

print(f"📥 Loading CLIP model: {CLIP_MODEL_NAME}")
print("   Model already downloaded, loading into memory...")

clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
clip_model = clip_model.to(DEVICE)
clip_model.eval()

with torch.no_grad():
    dummy_img = Image.new('RGB', (224, 224), color='white')
    dummy_inputs = clip_processor(images=dummy_img, return_tensors="pt")
    dummy_inputs = {k: v.to(DEVICE) for k, v in dummy_inputs.items()}

    dummy_output = clip_model.get_image_features(**dummy_inputs)

    if hasattr(dummy_output, 'shape'):
        EMBEDDING_DIM = dummy_output.shape[1]
    elif hasattr(dummy_output, 'pooler_output'):
        EMBEDDING_DIM = dummy_output.pooler_output.shape[1]
    elif hasattr(dummy_output, 'last_hidden_state'):
        EMBEDDING_DIM = dummy_output.last_hidden_state[:, 0, :].shape[1]
    else:
        EMBEDDING_DIM = 512
        print("   ⚠️ Using default embedding dim: 512")

total_params = sum(p.numel() for p in clip_model.parameters())

print(f"\n✅ Model loaded successfully!")
print(f"   Model: {CLIP_MODEL_NAME}")
print(f"   Device: {DEVICE}")
print(f"   Embedding dimension: {EMBEDDING_DIM}")
print(f"   Parameters: {total_params:,}")

print(f"\n{'='*55}")
print(f"✅ Section 6 COMPLETE — CLIP encoder ready")
print(f"{'='*55}")

In [ ]:
def generate_clip_embedding(image_pil, model, processor, device):
    """Generate normalized CLIP embedding for a PIL image."""
    try:
        inputs = processor(images=image_pil, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            features = model.get_image_features(**inputs)
        if hasattr(features, 'pooler_output'):
            features = features.pooler_output
        elif hasattr(features, 'last_hidden_state'):
            features = features.last_hidden_state[:, 0, :]
        features = features / features.norm(dim=-1, keepdim=True)
        return features.cpu().numpy().flatten()
    except:
        return None

print(f"Extracting CLIP embeddings for {len(subset_df):,} images...")
print(f"   Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print("   Expected time: ~10-15 minutes")
else:
    print("   Expected time: ~35-45 minutes (CPU)")

embeddings = []
valid_indices = []

for idx in tqdm(range(len(subset_df)), desc="Embedding Progress", unit="img"):
    row = subset_df.iloc[idx]
    cropped = crop_clothing(row['full_path'], row[BBOX_COL], padding=15)

    if cropped is None:
        try:
            cropped = Image.open(row['full_path']).convert("RGB")
        except:
            continue

    emb = generate_clip_embedding(cropped, clip_model, clip_processor, DEVICE)
    if emb is not None:
        embeddings.append(emb)
        valid_indices.append(idx)

embedding_matrix = np.array(embeddings, dtype=np.float32)
valid_df = subset_df.iloc[valid_indices].reset_index(drop=True)

print(f"\nSUCCESS!")
print(f"   Total embeddings: {len(embeddings):,}")
print(f"   Success rate:     {len(embeddings)/len(subset_df):.1%}")
print(f"   Matrix shape:     {embedding_matrix.shape}")
print(f"   Memory usage:     {embedding_matrix.nbytes / 1e6:.1f} MB")

print(f"\n{'='*60}")
print(f"SECTION 7 COMPLETE — Embeddings ready!")
print(f"{'='*60}")

In [ ]:
embedding_matrix = np.ascontiguousarray(embedding_matrix.astype(np.float32))
norms = np.linalg.norm(embedding_matrix, axis=1)

print("📐 EMBEDDING MATRIX DETAILS")
print("=" * 50)
print(f"   Shape:       {embedding_matrix.shape}")
print(f"   Dtype:       {embedding_matrix.dtype}")
print(f"   Memory:      {embedding_matrix.nbytes / 1e6:.1f} MB")
print(f"   Contiguous:  {embedding_matrix.flags['C_CONTIGUOUS']}")
print(f"   Norm range:  [{norms.min():.4f}, {norms.max():.4f}] (should be ~1.0)")
print(f"\n✅ Section 8 COMPLETE")

In [ ]:
print("🔨 Building FAISS IndexFlatL2...")

faiss_index = faiss.IndexFlatL2(EMBEDDING_DIM)
faiss_index.add(embedding_matrix)

print(f"   Vectors indexed: {faiss_index.ntotal:,}")
print(f"   Dimension:       {faiss_index.d}")

test_dist, test_idx = faiss_index.search(embedding_matrix[0:1], 3)
print(f"\n🧪 Test search (query=first image):")
print(f"   Nearest indices:  {test_idx[0]}")
print(f"   Distances:        {[f'{d:.4f}' for d in test_dist[0]]}")

print(f"\n✅ Section 9 COMPLETE — FAISS index ready")

In [ ]:
def search_similar(query_index, top_k=5):
    """
    Find top_k most similar clothing items.

    Args:
        query_index: Index in valid_df (0 to len(valid_df)-1)
        top_k: Number of results to return

    Returns:
        results: List of dicts with image info
        distances: Raw distance values
    """
    row = valid_df.iloc[query_index]

    cropped = crop_clothing(row['full_path'], row[BBOX_COL], padding=15)
    if cropped is None:
        cropped = Image.open(row['full_path']).convert("RGB")

    emb = generate_clip_embedding(cropped, clip_model, clip_processor, DEVICE)
    query_vec = emb.reshape(1, -1).astype(np.float32)

    distances, indices = faiss_index.search(query_vec, top_k + 1)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        if idx == query_index and dist < 0.001:
            continue 
        results.append({
            'index': int(idx),
            'distance': float(dist),
            'similarity': float(1.0 / (1.0 + dist)),
            'image_path': valid_df.iloc[idx]['full_path'],
            'bbox': valid_df.iloc[idx][BBOX_COL],
            'category': valid_df.iloc[idx]['category_name'],
        })

    return results[:top_k], distances[0]

print("🧪 Testing search function...")
test_results, _ = search_similar(0, top_k=3)
print(f"   Query: index 0 ({valid_df.iloc[0]['category_name']})")
print(f"   Found {len(test_results)} similar items:")
for r in test_results:
    print(f"      → Index {r['index']:4d} | {r['category']:20s} | dist={r['distance']:.4f}")

print(f"\n✅ Section 10 COMPLETE — Search function ready")

In [ ]:
def visualize_results(query_index, top_k=5):
    """Display query image and top-K similar items side by side."""

    query_row = valid_df.iloc[query_index]
    query_crop = crop_clothing(query_row['full_path'], query_row[BBOX_COL])

    results, _ = search_similar(query_index, top_k)

    n_cols = 1 + len(results)
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 5))

    if query_crop:
        axes[0].imshow(query_crop)
    axes[0].set_title(f"🔍 QUERY\n{query_row['category_name']}\nIndex: {query_index}",
                      fontsize=11, fontweight='bold', color='green')
    axes[0].axis('off')
    for spine in axes[0].spines.values():
        spine.set_visible(True)
        spine.set_color('green')
        spine.set_linewidth(3)

    for i, res in enumerate(results):
        crop = crop_clothing(res['image_path'], res['bbox'])
        if crop:
            axes[i + 1].imshow(crop)
        axes[i + 1].set_title(
            f"Rank #{i+1}\n{res['category']}\nDist: {res['distance']:.3f} | Sim: {res['similarity']:.1%}",
            fontsize=9
        )
        axes[i + 1].axis('off')

    plt.suptitle("👗 Clothing Similarity Search Results", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("🎯 SIMILARITY SEARCH EXAMPLES")
print("=" * 60)

example_queries = [0, 100, 500, 1500, 3000]
for q in example_queries:
    if q < len(valid_df):
        print(f"\n▶ Query #{q}: {valid_df.iloc[q]['category_name']}")
        visualize_results(q, top_k=4)

print(f"\n{'='*60}")
print(f"✅ Section 11 COMPLETE — Visual search working!")
print(f"{'='*60}")

In [ ]:
import os
import json
import faiss

os.makedirs("artifacts_fashion", exist_ok=True)

faiss.write_index(faiss_index, "artifacts_fashion/faiss.index")

np.save("artifacts_fashion/embeddings.npy", embedding_matrix)

valid_df[["full_path", "category_name"]].to_json(
    "artifacts_fashion/metadata.json",
    orient="records"
)

print("✅ artifacts saved")

In [ ]:
import os
from PIL import Image
from tqdm.auto import tqdm

THUMB_DIR = "fashion_thumbs"
os.makedirs(THUMB_DIR, exist_ok=True)

saved = 0

for i in tqdm(range(len(valid_df)), desc="Saving thumbnails"):
    row = valid_df.iloc[i]

    cropped = crop_clothing(row["full_path"], row[BBOX_COL], padding=15)

    if cropped is None:
        try:
            cropped = Image.open(row["full_path"]).convert("RGB")
        except:
            continue

    cropped = cropped.resize((224, 224))

    out_name = f"{i}.jpg"
    out_path = os.path.join(THUMB_DIR, out_name)
    cropped.save(out_path, quality=90)

    saved += 1

print("saved thumbnails:", saved)

In [ ]:
import os
print("thumb count:", len(os.listdir("fashion_thumbs")))
print("sample files:", os.listdir("fashion_thumbs")[:5])

In [ ]:
import shutil
shutil.make_archive("fashion_thumbs", "zip", "fashion_thumbs")
print("zip created")

In [ ]:
from IPython.display import FileLink
FileLink("fashion_thumbs.zip")